# 09 — Tier 2 Fidelity: Format Preservation, String Similarity, Cardinality

Verify that synthetic data preserves format patterns (emails, UUIDs), string distributions, and cardinality constraints.

In [ ]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.inference.tier2_profiler import run_tier2
import importlib
from sqllocks_spindle.cli import _get_domain_registry

spindle = Spindle()
reg = _get_domain_registry()
mod, cls, _ = reg['retail']
domain = getattr(importlib.import_module(mod), cls)(schema_mode='3nf')
real_result = spindle.generate(domain=domain, scale='small', seed=42)
synth_result = spindle.generate(domain=domain, scale='small', seed=99)
table = next(iter(real_result.tables))
print(f'Running Tier 2 on: {table}')

In [ ]:
report = run_tier2(real_result.tables[table], synth_result.tables[table])
print(report.summary())

In [ ]:
# Inspect per-column details
print('\n--- Format Preservation ---')
for col, r in report.format_preservation.items():
    status = 'PASS' if r.passed else 'FAIL'
    print(f'  [{status}] {col} ({r.detected_format}): real={r.real_format_rate:.1%}  synth={r.synth_format_rate:.1%}')

print('\n--- String Similarity (trigram cosine) ---')
for col, r in report.string_similarity.items():
    print(f'  {col}: {r.cosine_similarity:.4f}  score={r.score:.1f}/100')

print('\n--- Cardinality Constraints ---')
for col, r in report.cardinality.items():
    status = 'PASS' if r.passed else 'FAIL'
    print(f'  [{status}] {col}: real={r.real_cardinality}  synth={r.synth_cardinality}  ratio={r.ratio:.3f}')